# UMUD Challenge - Submission 3: Heavyweight Segmentation Pipeline
This notebook improves upon Submission 2 by:
1. Suppressing warnings to keep epoch outputs clean.
2. Using a powerful Pretrained Encoder (ResNet34) via `segmentation_models_pytorch`.
3. Using Albumentations for Heavy Data Augmentation (vital for ultrasound).
4. Increasing Image Size to 384x384 (captures fine fascicle details).
5. Using Test-Time Augmentation (TTA) during inference for a free accuracy boost.

**Instructions for Kaggle:**
1. Turn on the **GPU** and ensure **Internet** is enabled (needed to download the pretrained ResNet34 weights).
2. Run all cells.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore') # Clean up epoch outputs

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import glob
from tqdm.auto import tqdm
import math

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Configuration
# ==========================================
matching_dirs = glob.glob('/kaggle/input/**/apo_imgs_v1', recursive=True)
if matching_dirs:
    APO_IMG_DIR = Path(matching_dirs[0])
    COMPETITION_DIR = APO_IMG_DIR.parent
else:
    COMPETITION_DIR = Path(glob.glob('/kaggle/input/*')[0]) if glob.glob('/kaggle/input/*') else Path('.')
    APO_IMG_DIR = COMPETITION_DIR / 'apo_imgs_v1'

APO_MASK_DIR  = COMPETITION_DIR / 'apo_masks_v1'
FASC_MASK_DIR = COMPETITION_DIR / 'fasc_masks_v1'
TEST_DIR      = COMPETITION_DIR / 'test_images_v2'
OUT_DIR       = Path('/kaggle/working')
# ==========================================

IMG_SIZE = 384 # Increased from 256 to capture thin fascicle lines!
BATCH_SIZE = 8 # Reduced to fit 384x384 in Kaggle GPU
EPOCHS = 18 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALE_MM_PER_PIXEL = 0.06979


In [ ]:
# Albumentations transforms
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Emulate ultrasound speckle
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

class UltrasoundDataset(Dataset):
    def __init__(self, record_ids, transform=None):
        self.record_ids = record_ids
        self.transform = transform

    def __len__(self):
        return len(self.record_ids)

    def __getitem__(self, idx):
        rec = self.record_ids[idx]
        
        def find_file(directory, filename_stem):
            for p in directory.rglob('*'):
                if p.is_file() and filename_stem in p.stem and p.suffix.lower() in ['.tif', '.png', '.jpg']:
                    return str(p)
            return None

        img_path = find_file(APO_IMG_DIR, rec)
        apo_mask_path = find_file(APO_MASK_DIR, rec)
        fasc_mask_path = find_file(FASC_MASK_DIR, rec)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) if img_path else np.zeros((800, 1200), dtype=np.uint8)
        apo = cv2.imread(apo_mask_path, cv2.IMREAD_GRAYSCALE) if apo_mask_path else np.zeros((800, 1200), dtype=np.uint8)
        fasc = cv2.imread(fasc_mask_path, cv2.IMREAD_GRAYSCALE) if fasc_mask_path else np.zeros((800, 1200), dtype=np.uint8)

        # Convert masks to 0/1
        apo = (apo > 127).astype(np.float32)
        fasc = (fasc > 127).astype(np.float32)
        # Stack masks to (H, W, 2) for Albumentations
        masks = np.stack([apo, fasc], axis=-1)
        
        if self.transform:
            augmented = self.transform(image=img, mask=masks)
            img = augmented['image']
            masks = augmented['mask']
            # masks is (H, W, 2), convert to (2, H, W)
            masks = masks.permute(2, 0, 1)
        else:
            img = torch.from_numpy(img).unsqueeze(0).float()
            masks = torch.from_numpy(masks).permute(2, 0, 1).float()

        return img, masks

def get_clean_stems_recursive(directory):
    stems = set()
    for p in directory.rglob('*'):
        if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg']:
            stems.add(p.stem.replace('_apo', '').replace('_fasc', '').replace('_mask', ''))
    return stems

apo_imgs = get_clean_stems_recursive(APO_IMG_DIR)
apo_masks = get_clean_stems_recursive(APO_MASK_DIR)
fasc_masks = get_clean_stems_recursive(FASC_MASK_DIR)

valid_records = sorted(list(apo_imgs & (apo_masks | fasc_masks)))
val_size = max(1, int(len(valid_records) * 0.15))
train_records = valid_records[:-val_size]
val_records = valid_records[-val_size:]

train_ds = UltrasoundDataset(train_records, transform=train_transform)
val_ds = UltrasoundDataset(val_records, transform=val_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
# We use a ResNet34 encoder with U-Net decoder. Very powerful for medical images.
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=2,
).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda')

print(f"Starting SMP U-Net Training for {EPOCHS} epochs...")
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, masks in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False, disable=True): # Disable tqdm to avoid messy outputs
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
    
    train_loss /= len(train_ds)
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = criterion(preds, masks)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_ds)
    
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), OUT_DIR / 'best_unet_smp.pth')

print("Training Complete! Best Model Saved.")


In [ ]:
def measure_geometry(apo_mask, fasc_mask, scale_mm_per_pixel=0.06979):
    ### 1. PENNATION ANGLE (PA) ###
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fasc_mask, 8)
    angles, weights = [], []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 50: continue
        pts = np.column_stack(np.where(labels == i)).astype(np.float64)
        if len(pts) < 5: continue
        cov = np.cov(pts.T)
        if cov.shape != (2, 2): continue
        evals, evecs = np.linalg.eigh(cov)
        principal = evecs[:, np.argmax(evals)] 
        angle = np.degrees(np.arctan2(abs(principal[0]), abs(principal[1])))
        angles.append(angle)
        weights.append(float(area))
    pa_deg = np.average(angles, weights=weights) if angles else 15.0 
    
    ### 2. MUSCLE THICKNESS (MT) ###
    n_apo, apo_labels, apo_stats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    if n_apo >= 3:
        comps = sorted([(i, apo_stats[i, cv2.CC_STAT_AREA]) for i in range(1, n_apo)], key=lambda x: x[1], reverse=True)
        c1, c2 = comps[0][0], comps[1][0]
        y1, y2 = np.mean(np.where(apo_labels == c1)[0]), np.mean(np.where(apo_labels == c2)[0])
        sup_c, deep_c = (c1, c2) if y1 < y2 else (c2, c1)
        
        sup_mask = (apo_labels == sup_c).astype(np.uint8)
        deep_mask = (apo_labels == deep_c).astype(np.uint8)
        
        sup_y = np.argmax(sup_mask[::-1], axis=0)
        sup_bottom = np.where(sup_mask.any(axis=0), sup_mask.shape[0] - 1 - sup_y, np.nan)
        
        deep_top = np.where(deep_mask.any(axis=0), np.argmax(deep_mask, axis=0), np.nan)
        
        thickness_profile = deep_top - sup_bottom
        valid = ~np.isnan(thickness_profile) & (thickness_profile > 0)
        mt_mm = np.nanmedian(thickness_profile[valid]) * scale_mm_per_pixel if np.sum(valid) > 50 else 20.0
    else:
        mt_mm = 20.0
        
    ### 3. FASCICLE LENGTH (FL) ###
    fl_mm = mt_mm / math.sin(math.radians(pa_deg))
    
    pa_deg = np.clip(pa_deg, 3.0, 45.0)
    fl_mm = np.clip(fl_mm, 20.0, 200.0)
    mt_mm = np.clip(mt_mm, 5.0, 40.0)
    
    return pa_deg, fl_mm, mt_mm


In [ ]:
model.load_state_dict(torch.load(OUT_DIR / 'best_unet_smp.pth', weights_only=True))
model.eval()

results = []
test_files = sorted([p for p in TEST_DIR.rglob('*') if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg']])
print(f"Generating Heavyweight TTA Predictions for {len(test_files)} images...")

with torch.no_grad():
    for test_path in tqdm(test_files):
        img_orig = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
        if img_orig is None: continue
        orig_h, orig_w = img_orig.shape[:2]
        
        # Preprocess
        img_rs = cv2.resize(img_orig, (IMG_SIZE, IMG_SIZE))
        img_tensor = (img_rs.astype(np.float32) / 255.0 - 0.5) / 0.5
        img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0).to(DEVICE)
        
        # Test Time Augmentation (TTA): Predict normal and horizontally flipped
        with torch.amp.autocast('cuda'):
            # Normal
            preds_norm = torch.sigmoid(model(img_tensor))
            # Flipped
            img_tensor_flipped = torch.flip(img_tensor, dims=[3])
            preds_flipped = torch.sigmoid(model(img_tensor_flipped))
            preds_flipped = torch.flip(preds_flipped, dims=[3])
            
            # Average TTA predictions
            preds = (preds_norm + preds_flipped) / 2.0
            
        preds_np = (preds.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
        apo_pred, fasc_pred = preds_np[0], preds_np[1]
        
        apo_full = cv2.resize(apo_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        fasc_full = cv2.resize(fasc_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        
        pa, fl, mt = measure_geometry(apo_full, fasc_full)
        image_id = test_path.name if test_path.suffix else f"{test_path.stem}.tif"
        
        results.append({
            'image_id': image_id,
            'pa_deg': round(float(pa), 3),
            'fl_mm': round(float(fl), 3),
            'mt_mm': round(float(mt), 3)
        })

sub_df = pd.DataFrame(results)
sub_df.to_csv(OUT_DIR / 'submission.csv', sep=',', index=False)
print(f"Successfully generated Heavyweight submission.csv!")
print(sub_df.head())
